# 챗봇 - 검색기반

## 실행시 주의사항
### 1. 'ChatbotData.csv' 업로드 확인
* 실행 안됨
### 2. 'cached_emb_data.npy' 업로드 확인
* 없으면 새로 임베딩 시작 => 느림

In [42]:
# !pip install sentence-transformers

In [43]:
# 임포트
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer
import numpy as np
import pandas as pd
import os

In [44]:
# 데이터로드
df = pd.read_csv('ChatbotData.csv')
# df

In [45]:
# 모델 불러오기 : Sentence-Bert - 수많은 한국어 텍스트를 학습한 모델
model = SentenceTransformer('jhgan/ko-sroberta-multitask')


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: jhgan/ko-sroberta-multitask
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [46]:
# 코사인 유사도를 활용하기 위해
# 질문 데이터 임베딩
def load_emb_data(file_path):
  if os.path.exists(file_path):
    emb_data = np.load(file_path)
  else:
    print(f'{file_path} 파일이 없습니다.')
    print('새로 임베딩을 시작합니다.')
    print('질문 데이터 변환 중...')
    emb_data = model.encode(df['Q'])
    print('변환 완료!')
  return emb_data
emb_data = load_emb_data('cached_emb_data.npy')

In [47]:

# 함수 : 챗봇이 선택한 답변
def get_chatbot_response(user_input):
  # 코사인 유사도를 비교하기 위해 사용자 질문을 임베딩
  user_emb = model.encode([user_input])

  # 코사인 유사도를 계산
  cos_sims = cosine_similarity(user_emb, emb_data)

  # 가장 높은 유사도를 가진 인덱스(위치) 찾기
  best_idx = np.argmax(cos_sims)
  # 해당 인덱스에 있는 답변 반환
  # 선택된 답변
  answer = df.iloc[best_idx]['A']
  # 유사도
  score = cos_sims[0][best_idx]

  return answer, score


In [48]:
text = "오늘은 비가 오네요."
response, score = get_chatbot_response(text)
print(f'답변 : {response}({score*100:.1f}%)')

답변 : 그리고 멈출 거예요.(92.5%)


In [49]:
while True:
  text = input('대화 : ')

  # 종료 조건
  if text == '종료': break

  response, score = get_chatbot_response(text)
  print(f'답변 : {response}({score*100:.1f}%)')

대화 : 종료


In [50]:
# 임베딩 된 질문들을 저장(시간이 제일 오래 걸림)
np.save('cached_emb_data.npy', emb_data)

# 챗봇 - 생성기반

In [17]:
# !pip install transformers torch

In [18]:
# 임포트
import torch
from transformers import PreTrainedTokenizerFast, GPT2LMHeadModel
import re

In [19]:
# 모델 불러오기(kogpt2)
tokenizer = PreTrainedTokenizerFast.from_pretrained(
    "skt/kogpt2-base-v2",
    # bos_token : 문장의 시작을 알리는 기호
    # eos_token : 문장의 끝을 알리는 기호
    # unk_token : 알수없는 단어를 처리하는 기호
    # pad_token : 문장의 길이를 맞추기위해 넣는 기호
    # mask_token: 문장의 특정 단어를 가리고 맞히는 학습할 때 사용하는 기호
    bos_token='</s>', eos_token='</s>', unk_token='<unk>',
    pad_token='<pad>', mask_token='<mask>'
)
model = GPT2LMHeadModel.from_pretrained("skt/kogpt2-base-v2")

tokenizer.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/513M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/513M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/149 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie transformer.wte.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
GPT2LMHeadModel LOAD REPORT from: skt/kogpt2-base-v2
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...11}.attn.bias        | UNEXPECTED |  | 
transformer.h.{0...11}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [20]:
# 문장을 생성하는 생성
def generate_answer(prompt, max_len=50):

  # 입력 문장을 토큰화
  input_ids = tokenizer.encode(prompt, return_tensors="pt")

  # 이어질 문장을 생성
  output = model.generate(
      input_ids,
      max_length=max_len, # 생성할 문장의 최대 길이
      repetition_penalty=2.0, # 똑같은 말을 하면 벌칙
      do_sample=True, # False이면 가장 확률이 높은 단어를 고름 => 답변이 항상 같음
      top_k=50, # 단어 후보 중 상위 50개 안에서 고름
      top_p=0.95, # 후보 단어들의 누적 확률이 95% 되는 지점까지만 후보군에 포함
      pad_token_id=tokenizer.pad_token_id
  )
  # 생성된 토큰들을 한글 문장으로 복원
  return tokenizer.decode(output[0], skip_special_tokens=True)

In [21]:
# 생성한 답변 문장의 끝을 깔끔하게 정리하기 위한 함수
def clean_and_stop_generation(text, max_len=1000):
  # 답변을 .?!를 기준으로 나눔(한 문단을 문장으로 나눔)
  sentences = re.split(r'([.?!])', text)
  clean_text = ""
  # 아래 반복문에서 len(sentences) - 1 . -1를 하는 이유를 설명
  # => 문장의 끝(.?!)이 없는 문장은 버리기 위해
  # 문장 + 문장끝(.?!)를 붙이기 위해
  # 문장1. 문장2. 문장3(마무리 안됨)
  # [문장1, . , 문장2, . 문장3]
  # 0       1   2      3  4
  # 0 => 0 + 1,
  # 2 => 2 + 3
  # 4 => 4 + 5
  for i in range(0, len(sentences) - 1, 2):
    clean_text += sentences[i] + sentences[i+1]
    if len(clean_text) > max_len:
      break
  return clean_text if clean_text else text.strip()

In [22]:
def test_generate(text):
  answer = generate_answer(text)
  cleaned_answer = clean_and_stop_generation(answer, 50)
  print('-'*30)
  print(f"{cleaned_answer }")
  print('-'*30)

In [23]:
test_generate("오늘 비가 와요.")
test_generate("공부하기 싫어요")
test_generate("곧 점심 시간이에요")

------------------------------
오늘 비가 와요. 저는 그쪽을 보려고 한 건데 아마 제가 지금 말씀드리는 거죠. 아니 이게 내일 오전에 예상을 할 수 있는 사람이 안 보인다는 네네네.
------------------------------
------------------------------
공부하기 싫어요"라며 반문했다.
안 후보와 박 후보는 모두 안철수 전 서울대 융합과학기술대학원장 정책특보 출신의 대통령 비서실장에 대한 비판적 시각을 드러냈다.
------------------------------
------------------------------
곧 점심 시간이에요 ᄒ
저도 맥주는 한잔 마시게 되구요.
------------------------------


# 챗봇-RAG 기반

In [34]:
# 임포트

In [35]:
import pandas as pd
# 데이터 불러오기
df = pd.read_csv('ChatbotData.csv')

In [36]:
from transformers import PreTrainedTokenizerFast, GPT2LMHeadModel

# 검색 모델
ret_model = SentenceTransformer('jhgan/ko-sroberta-multitask')
# 생성 모델
tokenizer = PreTrainedTokenizerFast.from_pretrained(
    "skt/kogpt2-base-v2",
    bos_token='</s>', eos_token='</s>', unk_token='<unk>',
    pad_token='<pad>', mask_token='<mask>'
)
gen_model = GPT2LMHeadModel.from_pretrained("skt/kogpt2-base-v2")


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: jhgan/ko-sroberta-multitask
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/149 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie transformer.wte.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
GPT2LMHeadModel LOAD REPORT from: skt/kogpt2-base-v2
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...11}.attn.bias        | UNEXPECTED |  | 
transformer.h.{0...11}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [37]:
# 질문 데이터 임베딩
def load_emb_data(file_path):
  if os.path.exists(file_path):
    emb_data = np.load(file_path)
  else:
    print(f'{file_path} 파일이 없습니다.')
    print('새로 임베딩을 시작합니다.')
    print('질문 데이터 변환 중...')
    emb_data = model.encode(df['Q'])
    print('변환 완료!')
  return emb_data
emb_data = load_emb_data('cached_emb_data.npy')

In [51]:
def get_rag_chatbot_response(user_input, max_len=50):
  # 1. 검색기반으로 모범 답안 선택
  # 코사인 유사도를 비교하기 위해 사용자 질문을 임베딩
  user_emb = ret_model.encode([user_input])

  # 코사인 유사도를 계산
  cos_sims = cosine_similarity(user_emb, emb_data)

  # 가장 높은 유사도를 가진 인덱스(위치) 찾기
  best_idx = np.argmax(cos_sims)
  # 해당 인덱스에 있는 답변 반환
  # 선택된 답변
  ret_answer = df.iloc[best_idx]['A']

  # 2. 생성기반으로 모범 답안을 기준으로 이어서 답안 생성
  prompt = f'질문: {user_input}\n정보: {ret_answer}\n답변: {ret_answer},'

  # 입력 문장을 토큰화
  input_ids = tokenizer.encode(prompt, return_tensors="pt")

  # 이어질 문장을 생성
  output = gen_model.generate(
      input_ids,
      max_length=max_len, # 생성할 문장의 최대 길이
      repetition_penalty=2.0, # 똑같은 말을 하면 벌칙
      do_sample=True, # False이면 가장 확률이 높은 단어를 고름 => 답변이 항상 같음
      top_k=50, # 단어 후보 중 상위 50개 안에서 고름
      top_p=0.95, # 후보 단어들의 누적 확률이 95% 되는 지점까지만 후보군에 포함
      pad_token_id=tokenizer.pad_token_id
  )
  # 생성된 토큰들을 한글 문장으로 복원
  decoded = tokenizer.decode(output[0], skip_special_tokens=True)
  return clean_and_stop_generation(decoded, max_len)

In [52]:
# 생성한 답변 문장의 끝을 깔끔하게 정리하기 위한 함수
def clean_and_stop_generation(text, max_len=50):
  if "답변:" in text:
    # 답변:을 기준으로 분리 후 제일 마지막에 있는 내용을 가져옴
    # 질문: aaa\n정보:bbb\n답변:ccc
    text = text.split("답변: ")[-1].strip()

  sentences = re.split(r'([.?!])', text)
  clean_text = ""

  for i in range(0, len(sentences) - 1, 2):
    clean_text += sentences[i] + sentences[i+1]
    if len(clean_text) > max_len:
      break
  return clean_text if clean_text else text.strip()

In [57]:
def test_rag_generate(text):
  cleaned_answer = get_rag_chatbot_response(text,50)
  print('-'*30)
  print(f"{cleaned_answer }")
  print('-'*30)

In [58]:
test_rag_generate('점심을 먹어서 배부르다')
test_rag_generate('비가 오네요')
test_rag_generate('썸타고 있어요')
test_rag_generate('세상에서 젤 아픈 이야기는 뭐야')

------------------------------
즐거운 시간 보내시길 바랍니다.,
점심 먹기 전에 먹는 게 좋다는 의견에 따라 나온 얘기입니다.
------------------------------
------------------------------
그리고 멈출 거예요.,
( 잠드는 사이에 )
1. 정체가 불분명한 사람 말이오.
가운데의 한 쪽 팔을 이용해.
------------------------------
------------------------------
연애로 이어지길 바랄게요.,
김유빈 : (웃음) 네.
------------------------------
------------------------------
이뤄지지 못한 모든 사랑이야기죠., ~이미 다 끝났고,~ 이젠 끝나잖아요...
------------------------------
